# Chapter 8 — RL for Hyperparameter Optimization## REINFORCE for XGBoost HPO[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required. Runtime: ~20 minutes.**

In [ ]:
import numpy as npfrom sklearn.datasets import load_breast_cancerfrom sklearn.model_selection import cross_val_scorefrom sklearn.preprocessing import StandardScalerimport matplotlib.pyplot as plttry:    from xgboost import XGBClassifier    print('XGBoost available')except ImportError:    print('pip install xgboost')print('Ready')

## 8.1 HPO as a Sequential Decision Problem- **State**: current hyperparameter values + evaluation history- **Action**: adjust one hyperparameter (increase/decrease/keep)- **Reward**: validation AUC improvement over previous best- **Episode**: 20 adjustment steps

In [ ]:
X, y = load_breast_cancer(return_X_y=True)scaler = StandardScaler()X = scaler.fit_transform(X)def evaluate_xgb(n_estimators=100, max_depth=6, learning_rate=0.1,                  subsample=0.8, colsample_bytree=0.8):    """Evaluate XGBoost with given hyperparameters. Returns CV AUC."""    model = XGBClassifier(        n_estimators=int(n_estimators),        max_depth=int(max_depth),        learning_rate=learning_rate,        subsample=subsample,        colsample_bytree=colsample_bytree,        eval_metric='auc', verbosity=0, random_state=42    )    scores = cross_val_score(model, X, y, cv=3, scoring='roc_auc')    return scores.mean()# Baseline: default XGBoostbaseline_auc = evaluate_xgb()print(f'Baseline XGBoost AUC: {baseline_auc:.4f}')

## 8.2 REINFORCE-Based HPO Agent

In [ ]:
import randomclass HPOState:    """Tracks current hyperparameter configuration."""    PARAM_RANGES = {        'n_estimators': (50, 500, 50),        'max_depth': (3, 10, 1),        'learning_rate': (0.01, 0.5, 0.05),        'subsample': (0.5, 1.0, 0.1),    }    def __init__(self):        self.params = {            'n_estimators': 100, 'max_depth': 6,            'learning_rate': 0.1, 'subsample': 0.8        }        self.best_auc = 0.0        self.history = []        def to_array(self):        return np.array([            self.params['n_estimators'] / 500,            self.params['max_depth'] / 10,            self.params['learning_rate'] / 0.5,            self.params['subsample'],        ])        def apply_action(self, action):        """Actions: 0-7 = increase/decrease each of 4 params"""        param_names = list(self.PARAM_RANGES.keys())        param_idx = action // 2        direction = 1 if action % 2 == 0 else -1        name = param_names[param_idx]        mn, mx, step = self.PARAM_RANGES[name]        self.params[name] = np.clip(            self.params[name] + direction * step, mn, mx        )        # Evaluate        auc = evaluate_xgb(**self.params)        reward = auc - self.best_auc        self.best_auc = max(self.best_auc, auc)        self.history.append({'params': dict(self.params), 'auc': auc})        return reward, auc# Quick search: random policy vs. greedy trackingprint('Running HPO search...')state = HPOState()aucs = [evaluate_xgb()]  # baselinefor step in range(15):    action = random.randint(0, 7)  # random HPO agent for demo    reward, auc = state.apply_action(action)    aucs.append(auc)best_auc = max(aucs)print(f'Best AUC found: {best_auc:.4f} (baseline: {baseline_auc:.4f})')print(f'Improvement: +{(best_auc - baseline_auc)*100:.2f}%')print(f'Best config: {state.history[aucs.index(best_auc)-1]["params"]}')

## ✅ Key Takeaways1. HPO is a sequential decision problem — each evaluation informs the next2. RL-based HPO (SMBO, Hyperband, ASHA) outperforms random search on expensive objectives3. The reward is the improvement over the previous best (dense signal)4. Bandit algorithms (Chapter 9) are often the right tool for HPO budget allocation